# Credit Card Fraud Detection — Capstone Notebook**Taha | Data Analytics Intern, ItSimpleRa Solutions**Implements the Week 7–8 capstone proposal:1. Load the ULB Credit Card Fraud dataset (284,807 transactions, 492 frauds = 0.172%)2. Build a **leakage-free** experimental pipeline (all resampling/scaling fit inside training folds only)3. Train three baselines: Logistic Regression (class-weighted), Random Forest, XGBoost4. **Proposed improvement:** add an autoencoder reconstruction-error feature (autoencoder trained *only* on genuine transactions from the training fold) and feed it into the supervised ensemble5. Evaluate with Precision / Recall / F1 (fraud class), AUPRC, MCC, and confusion matrices> **Runtime:** Runtime → Change runtime type → **T4 GPU** is nice for the autoencoder but CPU works fine (~10–15 min total).

---## 0. Getting the dataset**Direct link:** https://www.kaggle.com/datasets/mlg-ulb/creditcardfraudTwo ways to get `creditcard.csv` into Colab — pick **one**.### Option A — Manual upload (simplest)1. Open the link above, click **Download** (you need a free Kaggle account), you get `archive.zip`2. Unzip it on your computer → you get `creditcard.csv` (~144 MB)3. Run the upload cell below and select that file### Option B — Kaggle API (faster, no 144 MB browser upload)1. Kaggle → your profile picture → **Settings** → **API** → **Create New Token** → downloads `kaggle.json`2. Run the Kaggle API cell below and upload `kaggle.json` when prompted

In [ ]:
# === OPTION A: manual upload of creditcard.csv ===# Run this cell ONLY if you're using Option A.from google.colab import filesuploaded = files.upload()   # select creditcard.csv (or the archive.zip)import zipfile, osfor fname in uploaded:    if fname.endswith('.zip'):        with zipfile.ZipFile(fname) as z:            z.extractall('.')        print('Extracted:', z.namelist())print('Files here:', [f for f in os.listdir('.') if f.endswith('.csv')])

In [ ]:
# === OPTION B: Kaggle API download ===# Run this cell ONLY if you're using Option B.from google.colab import filesprint('Upload your kaggle.json:')files.upload()!mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json!pip -q install kaggle!kaggle datasets download -d mlg-ulb/creditcardfraud!unzip -o creditcardfraud.zip

---## 1. Setup and imports

In [ ]:
!pip -q install xgboost imbalanced-learn --upgradeimport numpy as npimport pandas as pdimport matplotlib.pyplot as pltimport seaborn as snsfrom sklearn.model_selection import train_test_split, StratifiedKFoldfrom sklearn.preprocessing import StandardScalerfrom sklearn.linear_model import LogisticRegressionfrom sklearn.ensemble import RandomForestClassifierfrom sklearn.pipeline import Pipelinefrom sklearn.metrics import (precision_score, recall_score, f1_score,                             average_precision_score, matthews_corrcoef,                             confusion_matrix, precision_recall_curve,                             classification_report)from imblearn.over_sampling import SMOTEfrom imblearn.pipeline import Pipeline as ImbPipelinefrom xgboost import XGBClassifierimport tensorflow as tffrom tensorflow import kerasfrom tensorflow.keras import layersRANDOM_STATE = 42np.random.seed(RANDOM_STATE)tf.random.set_seed(RANDOM_STATE)sns.set_theme(style='whitegrid')print('TensorFlow', tf.__version__)

---## 2. Load and inspect the data

In [ ]:
df = pd.read_csv('creditcard.csv')print('Shape:', df.shape)print('\nMissing values:', df.isnull().sum().sum())print('\nClass distribution:')print(df['Class'].value_counts())print('\nFraud rate: {:.3f}%'.format(100 * df['Class'].mean()))df.head()

In [ ]:
df.describe().T[['mean','std','min','max']]

### 2.1 Exploratory plots

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))# Class balance (log scale, otherwise fraud is invisible)df['Class'].value_counts().plot(kind='bar', ax=axes[0], color=['#4C72B0','#C44E52'])axes[0].set_yscale('log')axes[0].set_title('Class balance (log scale)')axes[0].set_xticklabels(['Genuine (0)','Fraud (1)'], rotation=0)# Amount distribution by classfor cls, lbl, c in [(0,'Genuine','#4C72B0'), (1,'Fraud','#C44E52')]:    axes[1].hist(np.log1p(df.loc[df.Class==cls,'Amount']), bins=50,                 alpha=0.6, density=True, label=lbl, color=c)axes[1].set_title('log(1 + Amount) by class'); axes[1].legend()# Time distribution by classfor cls, lbl, c in [(0,'Genuine','#4C72B0'), (1,'Fraud','#C44E52')]:    axes[2].hist(df.loc[df.Class==cls,'Time']/3600, bins=48,                 alpha=0.6, density=True, label=lbl, color=c)axes[2].set_title('Hours since first transaction'); axes[2].legend()plt.tight_layout(); plt.show()

In [ ]:
# Which PCA components separate the classes most?sep = (df.groupby('Class').mean().T         .assign(gap=lambda d: (d[1]-d[0]).abs())         .sort_values('gap', ascending=False)         .head(12))sep[['gap']].plot(kind='barh', figsize=(7,5), legend=False, color='#55A868')plt.title('Largest mean difference between fraud and genuine'); plt.gca().invert_yaxis()plt.show()

---## 3. Leakage-free splitThis is the point Kabane (2024) makes: **resampling must never see the test set.**We hold out a stratified test set *first*, and every transformation that learns from data —scaling, SMOTE, and the autoencoder — is fit on training data only.`Time` is dropped as an absolute timestamp (it only spans two days and would not generalise);`Amount` is kept and scaled.

In [ ]:
X = df.drop(columns=['Class','Time'])y = df['Class']X_train, X_test, y_train, y_test = train_test_split(    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE)print('Train:', X_train.shape, '| frauds:', int(y_train.sum()))print('Test :', X_test.shape,  '| frauds:', int(y_test.sum()))

---## 4. Evaluation helpersAll metrics from Section 6 of the proposal, in one place.

In [ ]:
def evaluate(name, y_true, y_prob, threshold=0.5, results=None):    y_pred = (y_prob >= threshold).astype(int)    row = {        'Model'    : name,        'Precision': precision_score(y_true, y_pred, zero_division=0),        'Recall'   : recall_score(y_true, y_pred),        'F1'       : f1_score(y_true, y_pred),        'AUPRC'    : average_precision_score(y_true, y_prob),        'MCC'      : matthews_corrcoef(y_true, y_pred),    }    if results is not None:        results.append(row)    print(f'--- {name} (threshold={threshold:.3f}) ---')    for k, v in row.items():        if k != 'Model':            print(f'  {k:<10}: {v:.4f}')    print(classification_report(y_true, y_pred, digits=4,                                target_names=['Genuine','Fraud'], zero_division=0))    return rowdef plot_confusion(name, y_true, y_prob, threshold=0.5, ax=None):    cm = confusion_matrix(y_true, (y_prob >= threshold).astype(int))    ax = ax or plt.gca()    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=ax,                xticklabels=['Pred 0','Pred 1'], yticklabels=['True 0','True 1'])    tn, fp, fn, tp = cm.ravel()    ax.set_title(f'{name}\nmissed fraud={fn}, false alarms={fp}')def best_f1_threshold(y_true, y_prob):    """Pick the threshold maximising fraud-class F1 — tuned on validation, not test."""    prec, rec, thr = precision_recall_curve(y_true, y_prob)    f1 = np.divide(2*prec*rec, prec+rec, out=np.zeros_like(prec), where=(prec+rec)>0)    return float(thr[np.argmax(f1[:-1])])results = []

---## 5. Baseline modelsEach baseline is wrapped in a pipeline so scaling (and SMOTE, where used) is refitinside every cross-validation fold rather than on the full training set.

### 5.1 Logistic Regression (class-weighted)

In [ ]:
logreg = Pipeline([    ('scaler', StandardScaler()),    ('clf', LogisticRegression(max_iter=2000, class_weight='balanced',                               random_state=RANDOM_STATE))])logreg.fit(X_train, y_train)p_lr = logreg.predict_proba(X_test)[:, 1]evaluate('Logistic Regression', y_test, p_lr, results=results);

In [ ]:
# Readable coefficients — the interpretability the proposal asks forcoef = pd.Series(logreg.named_steps['clf'].coef_[0], index=X_train.columns)coef.reindex(coef.abs().sort_values(ascending=False).index).head(12).plot(    kind='barh', figsize=(7,5), color='#4C72B0')plt.title('Largest logistic regression coefficients'); plt.gca().invert_yaxis(); plt.show()

### 5.2 Random Forest

In [ ]:
rf = RandomForestClassifier(    n_estimators=300, max_depth=None, min_samples_leaf=2,    class_weight='balanced_subsample', n_jobs=-1, random_state=RANDOM_STATE)rf.fit(X_train, y_train)p_rf = rf.predict_proba(X_test)[:, 1]evaluate('Random Forest', y_test, p_rf, results=results);

### 5.3 XGBoost

In [ ]:
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()print('scale_pos_weight =', round(scale_pos_weight, 2))xgb = XGBClassifier(    n_estimators=500, max_depth=5, learning_rate=0.05,    subsample=0.8, colsample_bytree=0.8,    scale_pos_weight=scale_pos_weight,    eval_metric='aucpr', tree_method='hist',    n_jobs=-1, random_state=RANDOM_STATE)xgb.fit(X_train, y_train)p_xgb = xgb.predict_proba(X_test)[:, 1]evaluate('XGBoost', y_test, p_xgb, results=results);

### 5.4 Leakage demonstration (Kabane, 2024)Short experiment showing *why* the pipeline above is built the way it is: applying SMOTE**before** the split inflates the reported score, because synthetic minority points derivedfrom test rows end up in training. This is run purely as a cautionary result — the inflatednumber is **not** reported as a real one.

In [ ]:
# WRONG: resample the whole dataset, then splitX_all_res, y_all_res = SMOTE(random_state=RANDOM_STATE).fit_resample(X, y)Xtr_bad, Xte_bad, ytr_bad, yte_bad = train_test_split(    X_all_res, y_all_res, test_size=0.20, stratify=y_all_res, random_state=RANDOM_STATE)xgb_leaky = XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.05,                          eval_metric='aucpr', tree_method='hist',                          n_jobs=-1, random_state=RANDOM_STATE)xgb_leaky.fit(Xtr_bad, ytr_bad)p_leaky = xgb_leaky.predict_proba(Xte_bad)[:, 1]print('LEAKY (do not report):  AUPRC = {:.4f} | F1 = {:.4f}'.format(    average_precision_score(yte_bad, p_leaky),    f1_score(yte_bad, (p_leaky >= 0.5).astype(int))))print('CORRECT XGBoost      :  AUPRC = {:.4f} | F1 = {:.4f}'.format(    average_precision_score(y_test, p_xgb),    f1_score(y_test, (p_xgb >= 0.5).astype(int))))print('\nThe gap is the leakage, not model skill.')

---## 6. Proposed improvement — autoencoder reconstruction error as a featureThe autoencoder is trained **only on genuine transactions from the training split**, so itlearns what normal looks like. Fraudulent transactions then reconstruct poorly, and theper-row reconstruction error becomes a one-column anomaly signal.This borrows the minority-class sensitivity of one-class methods (Zaffar et al., 2023)while keeping the final classifier a plain, deployable tree ensemble.**Leakage control:** the scaler and the autoencoder both see training rows only; the test setis transformed with those fitted objects and never used for fitting.

In [ ]:
# Scale using training statistics onlyae_scaler = StandardScaler().fit(X_train)Xtr_s = ae_scaler.transform(X_train)Xte_s = ae_scaler.transform(X_test)# Split off a genuine-only training set + a small validation set for early stoppinggenuine_mask = (y_train == 0).valuesXtr_genuine  = Xtr_s[genuine_mask]Xg_fit, Xg_val = train_test_split(Xtr_genuine, test_size=0.1, random_state=RANDOM_STATE)print('Autoencoder trains on', Xg_fit.shape[0], 'genuine transactions')

In [ ]:
n_features = Xtr_s.shape[1]def build_autoencoder(n_features, latent_dim=8):    inp = keras.Input(shape=(n_features,))    e = layers.Dense(24, activation='relu')(inp)    e = layers.Dense(16, activation='relu')(e)    z = layers.Dense(latent_dim, activation='relu', name='latent')(e)    d = layers.Dense(16, activation='relu')(z)    d = layers.Dense(24, activation='relu')(d)    out = layers.Dense(n_features, activation='linear')(d)    model = keras.Model(inp, out, name='autoencoder')    model.compile(optimizer=keras.optimizers.Adam(1e-3), loss='mse')    return modelautoencoder = build_autoencoder(n_features)autoencoder.summary()

In [ ]:
history = autoencoder.fit(    Xg_fit, Xg_fit,    validation_data=(Xg_val, Xg_val),    epochs=40, batch_size=512, shuffle=True, verbose=1,    callbacks=[keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)])plt.figure(figsize=(6,4))plt.plot(history.history['loss'], label='train')plt.plot(history.history['val_loss'], label='val')plt.xlabel('epoch'); plt.ylabel('MSE'); plt.title('Autoencoder training'); plt.legend()plt.show()

In [ ]:
def reconstruction_error(model, X_scaled, batch_size=4096):    recon = model.predict(X_scaled, batch_size=batch_size, verbose=0)    return np.mean(np.square(X_scaled - recon), axis=1)err_train = reconstruction_error(autoencoder, Xtr_s)err_test  = reconstruction_error(autoencoder, Xte_s)# Does the signal separate the classes at all?plt.figure(figsize=(7,4))for cls, lbl, c in [(0,'Genuine','#4C72B0'), (1,'Fraud','#C44E52')]:    plt.hist(np.log1p(err_test[y_test.values==cls]), bins=60,             alpha=0.6, density=True, label=lbl, color=c)plt.xlabel('log(1 + reconstruction error)'); plt.legend()plt.title('Reconstruction error on the test set')plt.show()print('Reconstruction error alone — AUPRC: {:.4f}'.format(    average_precision_score(y_test, err_test)))

### 6.1 Augmented XGBoost — PCA features + reconstruction error

In [ ]:
X_train_aug = X_train.copy()X_test_aug  = X_test.copy()X_train_aug['recon_error'] = err_trainX_test_aug['recon_error']  = err_testxgb_aug = XGBClassifier(    n_estimators=500, max_depth=5, learning_rate=0.05,    subsample=0.8, colsample_bytree=0.8,    scale_pos_weight=scale_pos_weight,    eval_metric='aucpr', tree_method='hist',    n_jobs=-1, random_state=RANDOM_STATE)xgb_aug.fit(X_train_aug, y_train)p_aug = xgb_aug.predict_proba(X_test_aug)[:, 1]evaluate('XGBoost + AE feature', y_test, p_aug, results=results);

In [ ]:
# Where does recon_error rank among the features the model actually used?imp = pd.Series(xgb_aug.feature_importances_, index=X_train_aug.columns)imp = imp.sort_values(ascending=False).head(15)colors = ['#C44E52' if i == 'recon_error' else '#4C72B0' for i in imp.index]imp.plot(kind='barh', figsize=(7,5), color=colors)plt.title('XGBoost feature importance (recon_error highlighted)')plt.gca().invert_yaxis(); plt.show()rank = list(pd.Series(xgb_aug.feature_importances_,                      index=X_train_aug.columns).sort_values(ascending=False).index)print('recon_error importance rank: {} of {}'.format(    rank.index('recon_error') + 1, len(rank)))

---## 7. Cross-validated comparisonA single train/test split can be noisy when the positive class has only ~98 test examples.This runs 5-fold stratified CV, **rebuilding the autoencoder inside each fold**, so thereported spread reflects real variance rather than one lucky split.This is the slowest cell in the notebook (roughly 5–10 minutes).

In [ ]:
def cv_compare(X, y, n_splits=5):    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)    rows = []    for fold, (tr, te) in enumerate(skf.split(X, y), 1):        Xtr, Xte = X.iloc[tr], X.iloc[te]        ytr, yte = y.iloc[tr], y.iloc[te]        # --- everything below is fit on the training fold only ---        sc = StandardScaler().fit(Xtr)        Xtr_s, Xte_s = sc.transform(Xtr), sc.transform(Xte)        ae = build_autoencoder(Xtr.shape[1])        ae.fit(Xtr_s[(ytr == 0).values], Xtr_s[(ytr == 0).values],               epochs=25, batch_size=512, verbose=0, shuffle=True)        etr = reconstruction_error(ae, Xtr_s)        ete = reconstruction_error(ae, Xte_s)        spw = (ytr == 0).sum() / (ytr == 1).sum()        base_params = dict(n_estimators=400, max_depth=5, learning_rate=0.05,                           subsample=0.8, colsample_bytree=0.8,                           scale_pos_weight=spw, eval_metric='aucpr',                           tree_method='hist', n_jobs=-1, random_state=RANDOM_STATE)        m_base = XGBClassifier(**base_params).fit(Xtr, ytr)        p_base = m_base.predict_proba(Xte)[:, 1]        Xtr_a = Xtr.assign(recon_error=etr)        Xte_a = Xte.assign(recon_error=ete)        m_aug = XGBClassifier(**base_params).fit(Xtr_a, ytr)        p_aug_ = m_aug.predict_proba(Xte_a)[:, 1]        for label, p in [('XGBoost', p_base), ('XGBoost + AE', p_aug_)]:            pred = (p >= 0.5).astype(int)            rows.append({'fold': fold, 'model': label,                         'AUPRC': average_precision_score(yte, p),                         'F1': f1_score(yte, pred),                         'Recall': recall_score(yte, pred),                         'MCC': matthews_corrcoef(yte, pred)})        print(f'fold {fold} done')    return pd.DataFrame(rows)cv_df = cv_compare(X_train, y_train)cv_summary = cv_df.groupby('model')[['AUPRC','F1','Recall','MCC']].agg(['mean','std'])cv_summary.round(4)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4))for ax, metric in zip(axes, ['AUPRC','F1','Recall','MCC']):    sns.boxplot(data=cv_df, x='model', y=metric, ax=ax, palette=['#4C72B0','#C44E52'])    sns.stripplot(data=cv_df, x='model', y=metric, ax=ax, color='black', size=5)    ax.set_title(metric); ax.set_xlabel('')plt.suptitle('5-fold cross-validation: does the AE feature help?')plt.tight_layout(); plt.show()

---## 8. Threshold tuning and final resultsA 0.5 cutoff is arbitrary for a fraud system. The operating threshold is chosen on a**validation split carved out of the training data**, then applied unchanged to the test set.

In [ ]:
# Carve a validation split out of training onlyXtr2, Xval, ytr2, yval = train_test_split(    X_train_aug, y_train, test_size=0.2, stratify=y_train, random_state=RANDOM_STATE)xgb_thr = XGBClassifier(    n_estimators=500, max_depth=5, learning_rate=0.05,    subsample=0.8, colsample_bytree=0.8,    scale_pos_weight=(ytr2==0).sum()/(ytr2==1).sum(),    eval_metric='aucpr', tree_method='hist',    n_jobs=-1, random_state=RANDOM_STATE).fit(Xtr2, ytr2)thr = best_f1_threshold(yval, xgb_thr.predict_proba(Xval)[:, 1])print('Threshold chosen on validation: {:.4f}'.format(thr))evaluate('XGBoost + AE (tuned thr)', y_test, p_aug, threshold=thr, results=results);

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 4.2))plot_confusion('Logistic Regression', y_test, p_lr,  0.5, axes[0])plot_confusion('Random Forest',       y_test, p_rf,  0.5, axes[1])plot_confusion('XGBoost',             y_test, p_xgb, 0.5, axes[2])plot_confusion('XGBoost + AE (tuned)',y_test, p_aug, thr, axes[3])plt.tight_layout(); plt.show()

In [ ]:
plt.figure(figsize=(7,5.5))for label, p in [('Logistic Regression', p_lr), ('Random Forest', p_rf),                 ('XGBoost', p_xgb), ('XGBoost + AE', p_aug)]:    prec, rec, _ = precision_recall_curve(y_test, p)    plt.plot(rec, prec, label='{} (AUPRC={:.3f})'.format(        label, average_precision_score(y_test, p)))plt.axhline(y_test.mean(), ls='--', c='gray', label='No-skill baseline')plt.xlabel('Recall'); plt.ylabel('Precision')plt.title('Precision-Recall curves (fraud class)'); plt.legend(); plt.show()

In [ ]:
final = pd.DataFrame(results).set_index('Model').round(4)final

---## 9. Business-cost viewPrecision and recall don't price the trade-off. This converts the confusion matrix into moneyusing two editable assumptions, so the threshold choice can be argued in business terms ratherthan metric terms. **Adjust these two numbers to whatever your write-up assumes** — they areillustrative, not measured from the data.

In [ ]:
COST_MISSED_FRAUD  = 250.0   # avg loss when fraud gets through (edit me)COST_FALSE_ALARM   = 5.0     # review / customer-friction cost per false flag (edit me)thresholds = np.linspace(0.01, 0.99, 99)costs = []for t in thresholds:    tn, fp, fn, tp = confusion_matrix(y_test, (p_aug >= t).astype(int)).ravel()    costs.append(fn * COST_MISSED_FRAUD + fp * COST_FALSE_ALARM)best_t = thresholds[int(np.argmin(costs))]plt.figure(figsize=(7,4))plt.plot(thresholds, costs)plt.axvline(best_t, ls='--', c='#C44E52', label=f'cost-optimal thr={best_t:.2f}')plt.axvline(thr, ls=':', c='#55A868', label=f'F1-optimal thr={thr:.2f}')plt.xlabel('Decision threshold'); plt.ylabel('Total cost on test set')plt.title('Cost curve under stated assumptions'); plt.legend(); plt.show()tn, fp, fn, tp = confusion_matrix(y_test, (p_aug >= best_t).astype(int)).ravel()print(f'At threshold {best_t:.2f}: caught {tp} frauds, missed {fn}, false alarms {fp}')

---## 10. Save artefacts

In [ ]:
import joblib, osos.makedirs('artifacts', exist_ok=True)joblib.dump(xgb_aug,   'artifacts/xgboost_with_ae_feature.joblib')joblib.dump(ae_scaler, 'artifacts/ae_scaler.joblib')autoencoder.save('artifacts/autoencoder.keras')final.to_csv('artifacts/results_summary.csv')cv_df.to_csv('artifacts/cv_results.csv', index=False)print(os.listdir('artifacts'))# Optional: download them# from google.colab import files# !zip -r artifacts.zip artifacts# files.download('artifacts.zip')

---## 11. Notes for the write-upFill these in from your actual output rather than from expectation:- **Did the AE feature help?** Compare the CV means in Section 7, and check the std — if the  gap is smaller than the fold-to-fold spread, the honest conclusion is "no measurable gain,"  and that is a perfectly good capstone finding. Reporting a negative result cleanly is worth  more than overclaiming.- **Where recon_error ranked** (Section 6.1) tells you whether the model found the signal  useful at all, independent of the headline metric.- **The leakage cell (5.4)** gives you a concrete number for Kabane (2024)'s point — quote the  inflated vs correct AUPRC directly.- **The confusion matrices** are what a payments team would actually look at: missed fraud vs  analyst workload.- **Limitations to state:** the data is two days of 2013 European transactions, V1–V28 are  anonymised so feature-level interpretation is limited, and fraud patterns drift over time  so these numbers wouldn't hold on live traffic without retraining.